# Titanic Kaggle

In [1]:
import pandas as pd
import numpy as np

from sklearn.model_selection import StratifiedKFold, cross_val_score
from sklearn.pipeline import Pipeline
from sklearn.compose import ColumnTransformer
from sklearn.preprocessing import OneHotEncoder
from sklearn.impute import SimpleImputer
from sklearn.ensemble import GradientBoostingClassifier, RandomForestClassifier, ExtraTreesClassifier

pd.set_option('display.max_columns', 100)


## 1. Загрузка данных

In [2]:
train = pd.read_csv('train.csv')
test = pd.read_csv('test.csv')

print('Train shape:', train.shape)
print('Test shape:', test.shape)

display(train.head())


Train shape: (891, 12)
Test shape: (418, 11)


,PassengerId,Survived,Pclass,Name,Sex,Age,SibSp,Parch,Ticket,Fare,Cabin,Embarked
0,1,0,3,"Braund, Mr. Owen Harris",male,22.0,1,0,A/5 21171,7.2500,NaN,S
1,2,1,1,"Cumings, Mrs. John Bradley (Florence Briggs Th...",female,38.0,1,0,PC 17599,71.2833,C85,C
2,3,1,3,"Heikkinen, Miss. Laina",female,26.0,0,0,STON/O2. 3101282,7.9250,NaN,S
3,4,1,1,"Futrelle, Mrs. Jacques Heath (Lily May Peel)",female,35.0,1,0,113803,53.1000,C123,S
4,5,0,3,"Allen, Mr. William Henry",male,35.0,0,0,373450,8.0500,NaN,S


## 2. Быстрая проверка пропусков

In [3]:
missing = pd.DataFrame({
    'train_missing': train.isna().sum(),
    'test_missing': test.isna().sum()
})

display(missing)


,train_missing,test_missing
Age,177,86.0
Cabin,687,327.0
Embarked,2,0.0
Fare,0,1.0
Name,0,0.0
Parch,0,0.0
PassengerId,0,0.0
Pclass,0,0.0
Sex,0,0.0
SibSp,0,0.0


## 3. Feature Engineering

In [4]:
def add_features(train_df, test_df):
    train_df = train_df.copy()
    test_df = test_df.copy()

    full = pd.concat(
        [train_df.drop(columns=['Survived']), test_df],
        ignore_index=True
    )

    ticket_counts = full['Ticket'].value_counts()

    title_map = {
        'Mlle': 'Miss',
        'Ms': 'Miss',
        'Mme': 'Mrs',
        'Lady': 'Rare',
        'Countess': 'Rare',
        'Capt': 'Rare',
        'Col': 'Rare',
        'Don': 'Rare',
        'Dr': 'Rare',
        'Major': 'Rare',
        'Rev': 'Rare',
        'Sir': 'Rare',
        'Jonkheer': 'Rare',
        'Dona': 'Rare'
    }

    for df in [train_df, test_df]:
        df['Title'] = df['Name'].str.extract(r',\s*([^\.]+)\.', expand=False)
        df['Title'] = df['Title'].replace(title_map).fillna('Rare')

        df['FamilySize'] = df['SibSp'] + df['Parch'] + 1
        df['IsAlone'] = (df['FamilySize'] == 1).astype(int)
        df['SmallFamily'] = ((df['FamilySize'] >= 2) & (df['FamilySize'] <= 4)).astype(int)
        df['LargeFamily'] = (df['FamilySize'] >= 5).astype(int)

        df['CabinKnown'] = df['Cabin'].notna().astype(int)
        df['Deck'] = df['Cabin'].str[0].fillna('Unknown')

        df['TicketGroupSize'] = df['Ticket'].map(ticket_counts).fillna(1)

        df['FarePerPerson'] = df['Fare'] / df['FamilySize']

    return train_df, test_df


train_fe, test_fe = add_features(train, test)

display(train_fe.head())


,PassengerId,Survived,Pclass,Name,Sex,Age,SibSp,Parch,Ticket,Fare,Cabin,Embarked,Title,FamilySize,IsAlone,SmallFamily,LargeFamily,CabinKnown,Deck,TicketGroupSize,FarePerPerson
0,1,0,3,"Braund, Mr. Owen Harris",male,22.0,1,0,A/5 21171,7.2500,NaN,S,Mr,2,0,1,0,0,Unknown,1,3.62500
1,2,1,1,"Cumings, Mrs. John Bradley (Florence Briggs Th...",female,38.0,1,0,PC 17599,71.2833,C85,C,Mrs,2,0,1,0,1,C,2,35.64165
2,3,1,3,"Heikkinen, Miss. Laina",female,26.0,0,0,STON/O2. 3101282,7.9250,NaN,S,Miss,1,1,0,0,0,Unknown,1,7.92500
3,4,1,1,"Futrelle, Mrs. Jacques Heath (Lily May Peel)",female,35.0,1,0,113803,53.1000,C123,S,Mrs,2,0,1,0,1,C,2,26.55000
4,5,0,3,"Allen, Mr. William Henry",male,35.0,0,0,373450,8.0500,NaN,S,Mr,1,1,0,0,0,Unknown,1,8.05000


## 4. Подготовка X, y и test

`PassengerId`, `Name`, `Ticket`, `Cabin` убираем из модели. `PassengerId` нужен только для `submission.csv`.


In [6]:
TARGET = 'Survived'
ID_COL = 'PassengerId'

DROP_COLS = ['Name', 'Ticket', 'Cabin', ID_COL]

X = train_fe.drop(columns=[TARGET] + DROP_COLS)
y = train_fe[TARGET]

X_test = test_fe.drop(columns=DROP_COLS)

print('X shape:', X.shape)
print('X_test shape:', X_test.shape)

print('Columns:')
print(X.columns.tolist())


X shape: (891, 16)
X_test shape: (418, 16)
Columns:
['Pclass', 'Sex', 'Age', 'SibSp', 'Parch', 'Fare', 'Embarked', 'Title', 'FamilySize', 'IsAlone', 'SmallFamily', 'LargeFamily', 'CabinKnown', 'Deck', 'TicketGroupSize', 'FarePerPerson']


## 5. Разделение признаков на числовые и категориальные

In [7]:
numeric_features = X.select_dtypes(include=['number']).columns.tolist()
categorical_features = X.select_dtypes(exclude=['number']).columns.tolist()

print('Numeric:', numeric_features)
print('Categorical:', categorical_features)


Numeric: ['Pclass', 'Age', 'SibSp', 'Parch', 'Fare', 'FamilySize', 'IsAlone', 'SmallFamily', 'LargeFamily', 'CabinKnown', 'TicketGroupSize', 'FarePerPerson']
Categorical: ['Sex', 'Embarked', 'Title', 'Deck']


## 6. Preprocessing

Числовые признаки заполняем медианой, категориальные — самым частым значением и затем кодируем через OneHotEncoder.


In [8]:
numeric_transformer = Pipeline(steps=[
    ('imputer', SimpleImputer(strategy='median'))
])

categorical_transformer = Pipeline(steps=[
    ('imputer', SimpleImputer(strategy='most_frequent')),
    ('encoder', OneHotEncoder(handle_unknown='ignore'))
])

preprocessor = ColumnTransformer(
    transformers=[
        ('num', numeric_transformer, numeric_features),
        ('cat', categorical_transformer, categorical_features)
    ]
)


## 7. Сравнение нескольких моделей

На этих данных обычно сильнее всего работают аккуратный feature engineering + неглубокий бустинг/лес.  
Слишком сложная модель на Titanic легко переобучается.


In [9]:
models = {
    'GradientBoosting': GradientBoostingClassifier(
        n_estimators=100,
        learning_rate=0.05,
        max_depth=3,
        min_samples_leaf=5,
        subsample=0.8,
        random_state=42
    ),
    'RandomForest': RandomForestClassifier(
        n_estimators=200,
        max_depth=5,
        min_samples_leaf=3,
        max_features='sqrt',
        random_state=42,
        n_jobs=1
    ),
    'ExtraTrees': ExtraTreesClassifier(
        n_estimators=300,
        max_depth=6,
        min_samples_leaf=3,
        max_features='sqrt',
        random_state=42,
        n_jobs=1
    )
}

cv = StratifiedKFold(n_splits=5, shuffle=True, random_state=42)

scores = []

for model_name, model in models.items():
    pipe = Pipeline(steps=[
        ('preprocessor', preprocessor),
        ('model', model)
    ])

    cv_scores = cross_val_score(pipe, X, y, cv=cv, scoring='accuracy')

    scores.append({
        'model': model_name,
        'mean_accuracy': cv_scores.mean(),
        'std_accuracy': cv_scores.std()
    })

scores_df = pd.DataFrame(scores).sort_values('mean_accuracy', ascending=False)
display(scores_df)


,model,mean_accuracy,std_accuracy
0,GradientBoosting,0.839483,0.013347
1,RandomForest,0.832766,0.007571
2,ExtraTrees,0.831643,0.003847


## 8. Финальная модель

По проверке выше обычно лучше выходит `GradientBoostingClassifier`. Если у тебя в таблице выше победит другая модель, можешь заменить `final_model` на неё.


In [10]:
final_model = Pipeline(steps=[
    ('preprocessor', preprocessor),
    ('model', GradientBoostingClassifier(
        n_estimators=100,
        learning_rate=0.05,
        max_depth=3,
        min_samples_leaf=5,
        subsample=0.8,
        random_state=42
    ))
])

final_model.fit(X, y)

print('Final model fitted successfully.')


Final model fitted successfully.


## 9. Предсказание для test.csv

In [11]:
test_predictions = final_model.predict(X_test)

test_predictions[:20]


array([0, 0, 0, 0, 1, 0, 1, 0, 1, 0, 0, 0, 1, 0, 1, 1, 0, 0, 1, 1])

## 10. Создание submission.csv

In [12]:
submission = pd.DataFrame({
    'PassengerId': test[ID_COL],
    'Survived': test_predictions.astype(int)
})

submission.to_csv('submission.csv', index=False)

display(submission.head())
print('Saved: submission.csv')
print('Submission shape:', submission.shape)


,PassengerId,Survived
0,892,0
1,893,0
2,894,0
3,895,0
4,896,1


Saved: submission.csv
Submission shape: (418, 2)
